# Alert Health Monitor Analysis Examples

This notebook demonstrates how to use the Alert Health Monitor system for analyzing and optimizing your Sentry alert rules.

## Setup

First, let's import the necessary libraries and configure our environment.

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))

import httpx
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import json

# Configuration
API_BASE_URL = "http://localhost:8000/api/v1"
SENTRY_API_KEY = os.getenv("SENTRY_API_KEY", "your-sentry-api-key")

# Configure plotting
plt.style.use('ggplot')
sns.set_palette("husl")

print("Alert Health Monitor Analysis Notebook")
print(f"API URL: {API_BASE_URL}")

## 1. Health Metrics Analysis

Let's start by analyzing the health metrics of our alert rules.

In [ ]:
async def get_health_metrics(start_date, end_date, rule_ids=None):
    """Fetch health metrics for alert rules."""
    async with httpx.AsyncClient() as client:
        payload = {
            "start_date": start_date.isoformat() + "Z",
            "end_date": end_date.isoformat() + "Z",
            "include_patterns": True
        }
        if rule_ids:
            payload["rule_ids"] = rule_ids
            
        response = await client.post(
            f"{API_BASE_URL}/alert-health/metrics",
            json=payload,
            headers={"X-Sentry-Auth": SENTRY_API_KEY}
        )
        return response.json() if response.status_code == 200 else None

# Get metrics for the last 30 days
end_date = datetime.utcnow()
start_date = end_date - timedelta(days=30)

metrics_data = await get_health_metrics(start_date, end_date)

if metrics_data:
    # Convert to DataFrame for analysis
    metrics_df = pd.DataFrame(metrics_data['metrics'])
    print(f"Analyzed {len(metrics_df)} alert rules")
    print("\nTop 5 noisiest rules:")
    noisy_rules = metrics_df.nlargest(5, 'noise_score')[['rule_name', 'noise_score', 'false_positive_rate']]
    print(noisy_rules)

## 2. Visualizing Alert Patterns

Let's visualize the alert patterns to identify problematic rules.

In [ ]:
# Visualize noise levels across rules
if metrics_data and len(metrics_df) > 0:
    plt.figure(figsize=(12, 6))
    
    # Noise score distribution
    plt.subplot(1, 2, 1)
    metrics_df['noise_score'].hist(bins=20, edgecolor='black')
    plt.xlabel('Noise Score')
    plt.ylabel('Number of Rules')
    plt.title('Distribution of Noise Scores')
    
    # False positive rate vs trigger count
    plt.subplot(1, 2, 2)
    plt.scatter(metrics_df['total_triggered_count'], 
                metrics_df['false_positive_rate'],
                alpha=0.6)
    plt.xlabel('Total Triggered Count')
    plt.ylabel('False Positive Rate')
    plt.title('False Positives vs Alert Frequency')
    plt.xscale('log')
    
    plt.tight_layout()
    plt.show()

## 3. Alert Storm Detection

Now let's detect and analyze alert storms.

In [ ]:
async def detect_storms(start_date, end_date, min_alerts=50):
    """Detect alert storms in the specified time period."""
    async with httpx.AsyncClient() as client:
        response = await client.post(
            f"{API_BASE_URL}/alert-health/storms",
            json={
                "start_date": start_date.isoformat() + "Z",
                "end_date": end_date.isoformat() + "Z",
                "min_alerts_threshold": min_alerts
            },
            headers={"X-Sentry-Auth": SENTRY_API_KEY}
        )
        return response.json() if response.status_code == 200 else None

# Detect storms
storms_data = await detect_storms(start_date, end_date)

if storms_data:
    storms_df = pd.DataFrame(storms_data['storms'])
    print(f"Detected {len(storms_df)} alert storms")
    
    if len(storms_df) > 0:
        # Storm severity distribution
        plt.figure(figsize=(10, 6))
        
        severity_counts = storms_df['severity'].value_counts()
        plt.subplot(1, 2, 1)
        severity_counts.plot(kind='bar')
        plt.xlabel('Storm Severity')
        plt.ylabel('Count')
        plt.title('Alert Storm Severity Distribution')
        
        # Storm duration vs alerts
        plt.subplot(1, 2, 2)
        plt.scatter(storms_df['duration_minutes'], 
                    storms_df['total_alerts'],
                    c=storms_df['severity'].map({'low': 'green', 'medium': 'orange', 'high': 'red'}),
                    alpha=0.6)
        plt.xlabel('Duration (minutes)')
        plt.ylabel('Total Alerts')
        plt.title('Storm Duration vs Alert Count')
        
        plt.tight_layout()
        plt.show()

## 4. Threshold Optimization

Let's get recommendations for optimizing alert thresholds.

In [ ]:
async def get_threshold_recommendations(rule_ids, optimization_goal='balanced'):
    """Get threshold recommendations for specific rules."""
    async with httpx.AsyncClient() as client:
        response = await client.post(
            f"{API_BASE_URL}/alert-health/recommendations",
            json={
                "rule_ids": rule_ids,
                "optimization_goal": optimization_goal,
                "min_confidence": 0.7
            },
            headers={"X-Sentry-Auth": SENTRY_API_KEY}
        )
        return response.json() if response.status_code == 200 else None

# Get recommendations for the noisiest rules
if metrics_data and len(metrics_df) > 0:
    noisy_rule_ids = metrics_df.nlargest(5, 'noise_score')['rule_id'].tolist()
    recommendations = await get_threshold_recommendations(noisy_rule_ids)
    
    if recommendations:
        recs_df = pd.DataFrame(recommendations['recommendations'])
        
        # Visualize potential improvements
        plt.figure(figsize=(10, 6))
        
        # Expected noise reduction
        recs_df['expected_noise_reduction'] = recs_df['expected_impact'].apply(lambda x: x['noise_reduction'])
        recs_df.sort_values('expected_noise_reduction', ascending=True).plot(
            x='rule_id', 
            y='expected_noise_reduction',
            kind='barh',
            figsize=(10, 6)
        )
        plt.xlabel('Expected Noise Reduction')
        plt.title('Threshold Optimization Opportunities')
        plt.tight_layout()
        plt.show()
        
        print("\nTop Recommendations:")
        for _, rec in recs_df.iterrows():
            print(f"Rule: {rec['rule_id']}")
            print(f"  Current: {rec['current_threshold']} → Recommended: {rec['recommended_threshold']}")
            print(f"  Expected noise reduction: {rec['expected_noise_reduction']:.1%}")
            print(f"  Confidence: {rec['confidence_score']:.2f}\n")

## 5. Dashboard View

Finally, let's create a comprehensive dashboard view.

In [ ]:
async def get_dashboard_data(start_date, end_date):
    """Get comprehensive dashboard data."""
    async with httpx.AsyncClient() as client:
        response = await client.get(
            f"{API_BASE_URL}/alert-health/dashboard",
            params={
                "start_date": start_date.isoformat() + "Z",
                "end_date": end_date.isoformat() + "Z"
            },
            headers={"X-Sentry-Auth": SENTRY_API_KEY}
        )
        return response.json() if response.status_code == 200 else None

# Get dashboard data
dashboard_data = await get_dashboard_data(start_date, end_date)

if dashboard_data:
    # Create dashboard visualization
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))
    
    # Overall health score
    ax1.text(0.5, 0.5, f"{dashboard_data['overall_health_score']:.1f}", 
             fontsize=48, ha='center', va='center')
    ax1.set_title('Overall Health Score', fontsize=16)
    ax1.set_xlim(0, 1)
    ax1.set_ylim(0, 1)
    ax1.axis('off')
    
    # Noisy rules
    noisy_rules_df = pd.DataFrame(dashboard_data['noisy_rules'][:5])
    if len(noisy_rules_df) > 0:
        noisy_rules_df.plot(x='rule_name', y='noise_score', kind='bar', ax=ax2)
        ax2.set_title('Top 5 Noisiest Rules', fontsize=16)
        ax2.set_xlabel('Rule')
        ax2.set_ylabel('Noise Score')
    
    # Recent storms timeline
    storms_timeline = pd.DataFrame(dashboard_data['recent_storms'])
    if len(storms_timeline) > 0:
        storms_timeline['start_time'] = pd.to_datetime(storms_timeline['start_time'])
        storms_timeline.set_index('start_time')['total_alerts'].plot(kind='line', ax=ax3)
        ax3.set_title('Recent Alert Storms', fontsize=16)
        ax3.set_xlabel('Time')
        ax3.set_ylabel('Alert Count')
    
    # Optimization opportunities
    opt_df = pd.DataFrame(dashboard_data['optimization_opportunities'][:5])
    if len(opt_df) > 0:
        opt_df['potential_noise_reduction'] = opt_df['potential_noise_reduction'] * 100
        opt_df.plot(x='rule_name', y='potential_noise_reduction', kind='bar', ax=ax4)
        ax4.set_title('Top Optimization Opportunities', fontsize=16)
        ax4.set_xlabel('Rule')
        ax4.set_ylabel('Potential Noise Reduction (%)')
    
    plt.suptitle('Alert Health Monitor Dashboard', fontsize=20)
    plt.tight_layout()
    plt.show()
    
    # Print key stats
    print(f"\nKey Statistics:")
    print(f"Total rules analyzed: {dashboard_data['total_rules']}")
    print(f"Active rules: {dashboard_data['active_rules']}")
    print(f"Total alerts: {dashboard_data['total_alerts']}")
    print(f"Alert storms detected: {len(dashboard_data['recent_storms'])}")